In [2]:
import random
import json
import os
import pandas as pd
import numpy as np
from pathlib import Path

In [3]:
BASE_CATALOG = {
    "Network": [
        {
            "issue_type": "VPN Authentication Failure",
            "priority": "High",
            "resolution": "Clear saved VPN credentials, re-enter password, and reconnect.",
            "subject_templates": [
                "VPN not connecting after password reset",
                "Unable to login to VPN",
                "VPN authentication failed"
            ],
            "description_templates": [
                "After resetting my password, VPN is not connecting and shows authentication failed.",
                "I cannot connect to VPN since morning. It says invalid credentials.",
                "VPN login keeps failing even after entering the correct password."
            ]
        },
        {
            "issue_type": "WiFi Not Connecting",
            "priority": "Medium",
            "resolution": "Restart WiFi adapter, forget network, and reconnect.",
            "subject_templates": [
                "Laptop not connecting to WiFi",
                "Office WiFi not working",
                "Unable to join wireless network"
            ],
            "description_templates": [
                "My laptop cannot connect to office WiFi and keeps disconnecting.",
                "WiFi is visible but connection fails every time.",
                "Wireless network is not working on my device."
            ]
        },
        {
            "issue_type": "DNS Resolution Error",
            "priority": "Medium",
            "resolution": "Flush DNS cache and check network settings.",
            "subject_templates": [
                "DNS lookup failed",
                "Cannot resolve host",
                "Website not opening due to DNS issue"
            ],
            "description_templates": [
                "Internal websites are not opening and DNS lookup is failing.",
                "I am getting host resolution errors while browsing.",
                "System shows DNS server not responding."
            ]
        }
    ],

    "Email": [
        {
            "issue_type": "Outlook Password Prompt Loop",
            "priority": "High",
            "resolution": "Clear saved Outlook credentials and sign in again.",
            "subject_templates": [
                "Outlook keeps asking for password",
                "Password prompt appears again and again in Outlook",
                "Outlook login loop issue"
            ],
            "description_templates": [
                "Outlook keeps asking for password every few minutes.",
                "Even after entering the password, Outlook shows the login popup again.",
                "My mailbox is not syncing because Outlook repeatedly asks for credentials."
            ]
        },
        {
            "issue_type": "Mailbox Sync Issue",
            "priority": "Medium",
            "resolution": "Restart Outlook and re-sync the mailbox.",
            "subject_templates": [
                "Mailbox not syncing",
                "Emails are not updating in Outlook",
                "Inbox sync problem"
            ],
            "description_templates": [
                "New emails are not appearing in Outlook since yesterday.",
                "My mailbox is not syncing properly and sent emails are delayed.",
                "Inbox and sent items are not updating."
            ]
        },
        {
            "issue_type": "Email Sending Failure",
            "priority": "High",
            "resolution": "Check SMTP settings and mailbox quota.",
            "subject_templates": [
                "Unable to send email",
                "Outgoing mail failure",
                "Email stuck in outbox"
            ],
            "description_templates": [
                "Emails are not being sent and remain in outbox.",
                "Outlook shows sending failed error for all outgoing emails.",
                "I cannot send mail to clients since morning."
            ]
        }
    ],

    "Hardware": [
        {
            "issue_type": "Laptop Not Powering On",
            "priority": "High",
            "resolution": "Check charger connection, battery status, and perform hard reboot.",
            "subject_templates": [
                "Laptop not turning on",
                "System not powering on",
                "Black screen on startup"
            ],
            "description_templates": [
                "My laptop does not turn on even after pressing power button.",
                "System shows black screen and does not boot.",
                "Laptop is completely unresponsive after shutdown."
            ]
        },
        {
            "issue_type": "Keyboard Not Working",
            "priority": "Medium",
            "resolution": "Reconnect keyboard driver or restart the device.",
            "subject_templates": [
                "Laptop keyboard not working",
                "Keys are not responding",
                "Keyboard input issue"
            ],
            "description_templates": [
                "Several keys on my keyboard are not responding.",
                "Keyboard stopped working after restart.",
                "I cannot type because keyboard input is failing."
            ]
        },
        {
            "issue_type": "Blue Screen Error",
            "priority": "High",
            "resolution": "Collect error code, reboot system, and run diagnostics.",
            "subject_templates": [
                "Blue screen on laptop",
                "System crashed with blue screen",
                "Windows stop error"
            ],
            "description_templates": [
                "My system suddenly crashed and showed a blue screen.",
                "Laptop restarts after displaying a Windows stop error.",
                "I got a blue screen while working on a document."
            ]
        }
    ],

    "Software": [
        {
            "issue_type": "Application Crash",
            "priority": "Medium",
            "resolution": "Restart the application, clear cache, and reinstall if needed.",
            "subject_templates": [
                "Application keeps crashing",
                "Software closes unexpectedly",
                "App crash issue"
            ],
            "description_templates": [
                "The application closes immediately after opening.",
                "Software crashes whenever I try to use a feature.",
                "App stops responding and shuts down."
            ]
        },
        {
            "issue_type": "Software Installation Failure",
            "priority": "Medium",
            "resolution": "Run installer as admin and verify dependencies.",
            "subject_templates": [
                "Software installation failed",
                "Cannot install application",
                "Installer error"
            ],
            "description_templates": [
                "The software installer fails before completion.",
                "I get an error while installing the application.",
                "Setup wizard exits with installation failed message."
            ]
        },
        {
            "issue_type": "License Activation Error",
            "priority": "High",
            "resolution": "Verify license key and internet connectivity.",
            "subject_templates": [
                "License activation failed",
                "Software not activating",
                "Invalid license error"
            ],
            "description_templates": [
                "The application shows activation failed after entering license key.",
                "Software says the license is invalid or expired.",
                "I cannot activate the product on my laptop."
            ]
        }
    ],

    "Access": [
        {
            "issue_type": "Shared Folder Access Denied",
            "priority": "High",
            "resolution": "Verify folder permissions and user access rights.",
            "subject_templates": [
                "Cannot access shared folder",
                "Access denied on network folder",
                "Shared drive permission issue"
            ],
            "description_templates": [
                "I get access denied when opening the shared folder.",
                "The network drive opens for others but not for me.",
                "I do not have permission to access team shared files."
            ]
        },
        {
            "issue_type": "Account Locked",
            "priority": "High",
            "resolution": "Unlock the account and reset password if required.",
            "subject_templates": [
                "My account is locked",
                "Unable to login due to account lock",
                "User account locked out"
            ],
            "description_templates": [
                "I cannot login because the system says my account is locked.",
                "Too many failed login attempts locked my account.",
                "Please unlock my account so I can access office systems."
            ]
        },
        {
            "issue_type": "SSO Login Failure",
            "priority": "High",
            "resolution": "Check identity provider status and reauthenticate user.",
            "subject_templates": [
                "SSO login not working",
                "Cannot sign in using single sign-on",
                "Authentication redirect error"
            ],
            "description_templates": [
                "Single sign-on login fails after redirecting to the auth page.",
                "I cannot access the app through SSO.",
                "SSO authentication keeps failing with login error."
            ]
        }
    ],

    "Printer": [
        {
            "issue_type": "Printer Offline",
            "priority": "Medium",
            "resolution": "Restart printer and verify network connection.",
            "subject_templates": [
                "Printer showing offline",
                "Unable to print because printer is offline",
                "Office printer not available"
            ],
            "description_templates": [
                "The office printer is offline and I cannot print documents.",
                "Printer status shows offline on my laptop.",
                "Printing is blocked because the printer is unavailable."
            ]
        },
        {
            "issue_type": "Print Queue Stuck",
            "priority": "Low",
            "resolution": "Clear print queue and restart print spooler service.",
            "subject_templates": [
                "Print jobs stuck in queue",
                "Printer queue not clearing",
                "Documents not printing"
            ],
            "description_templates": [
                "My print jobs are stuck and nothing is printing.",
                "Documents remain in the print queue for a long time.",
                "Printer queue is blocked and jobs are not moving."
            ]
        },
        {
            "issue_type": "Driver Missing",
            "priority": "Medium",
            "resolution": "Install or update the printer driver.",
            "subject_templates": [
                "Printer driver missing",
                "Cannot detect printer driver",
                "Printer setup issue"
            ],
            "description_templates": [
                "The system says printer driver is missing.",
                "Printer is connected but driver is not installed.",
                "I cannot print because the required printer driver is unavailable."
            ]
        }
    ],

    "Database": [
        {
            "issue_type": "Database Connection Timeout",
            "priority": "High",
            "resolution": "Check DB server availability and connection string.",
            "subject_templates": [
                "Database connection timeout",
                "App cannot connect to database",
                "DB timeout error"
            ],
            "description_templates": [
                "The application times out while connecting to the database.",
                "Database connection fails after waiting for a long time.",
                "I am getting timeout errors while accessing DB-backed app."
            ]
        },
        {
            "issue_type": "Authentication to DB Failed",
            "priority": "High",
            "resolution": "Verify DB credentials and access permissions.",
            "subject_templates": [
                "Database login failed",
                "Cannot authenticate to DB",
                "DB credential error"
            ],
            "description_templates": [
                "Database connection fails due to authentication error.",
                "The app cannot login to the database using configured credentials.",
                "I am seeing invalid database username or password error."
            ]
        },
        {
            "issue_type": "Database Service Down",
            "priority": "High",
            "resolution": "Restart DB service and inspect server logs.",
            "subject_templates": [
                "Database service is down",
                "DB server not responding",
                "Application unavailable due to database outage"
            ],
            "description_templates": [
                "The database server appears to be down.",
                "Applications are failing because the DB service is unavailable.",
                "I cannot access data because the backend database is not responding."
            ]
        }
    ],

    "Security": [
        {
            "issue_type": "Antivirus Update Failed",
            "priority": "Medium",
            "resolution": "Check update server connectivity and retry update.",
            "subject_templates": [
                "Antivirus update failed",
                "Security definitions not updating",
                "Endpoint protection update issue"
            ],
            "description_templates": [
                "The antivirus software failed to download latest updates.",
                "Security definitions are not updating on my laptop.",
                "Endpoint protection shows update failed error."
            ]
        },
        {
            "issue_type": "MFA Code Not Received",
            "priority": "High",
            "resolution": "Check registered device and resend MFA code.",
            "subject_templates": [
                "Not receiving MFA code",
                "OTP not coming for login",
                "Two-factor authentication issue"
            ],
            "description_templates": [
                "I am not receiving the MFA code on my phone.",
                "OTP is not delivered during login verification.",
                "Two-factor authentication fails because code never arrives."
            ]
        },
        {
            "issue_type": "Endpoint Threat Detected",
            "priority": "High",
            "resolution": "Isolate device and run full antivirus scan.",
            "subject_templates": [
                "Threat detected on laptop",
                "Security software found malware",
                "Endpoint protection alert"
            ],
            "description_templates": [
                "Endpoint protection detected a threat on my system.",
                "The antivirus tool flagged malware on my laptop.",
                "Security alert says suspicious file was found on device."
            ]
        }
    ]
}

In [4]:
CATEGORY_VARIATIONS = {
    "Network": ["router", "gateway", "proxy", "lan", "wan", "wireless", "ethernet", "firewall"],
    "Email": ["outlook", "exchange", "mailbox", "smtp", "imap", "calendar", "outbox", "inbox"],
    "Hardware": ["laptop", "desktop", "monitor", "keyboard", "battery", "fan", "screen", "dock"],
    "Software": ["app", "tool", "installer", "license", "browser", "excel", "erp", "client"],
    "Access": ["folder", "vpn portal", "dashboard", "shared drive", "sso", "account", "admin panel", "hr portal"],
    "Printer": ["printer", "scanner", "spooler", "toner", "queue", "duplex", "tray", "label printer"],
    "Database": ["mysql", "postgres", "oracle", "sql server", "mongodb", "warehouse", "report db", "analytics db"],
    "Security": ["mfa", "otp", "endpoint", "antivirus", "defender", "login alert", "security agent", "compliance tool"]
}

In [5]:
def clone_issue(base_issue, category, clone_index):
    asset = random.choice(CATEGORY_VARIATIONS[category])

    new_issue_type = f"{base_issue['issue_type']} Variant {clone_index}"
    new_subjects = [f"{s} on {asset}" for s in base_issue["subject_templates"]]
    new_descriptions = [f"{d} This is affecting {asset} usage." for d in base_issue["description_templates"]]

    return {
        "issue_type": new_issue_type,
        "priority": base_issue["priority"],
        "resolution": base_issue["resolution"],
        "subject_templates": new_subjects,
        "description_templates": new_descriptions
    }


def generate_large_catalog(target_issue_types=5000):
    categories = list(BASE_CATALOG.keys())
    catalog = {category: [] for category in categories}

    for category, issues in BASE_CATALOG.items():
        for issue in issues:
            catalog[category].append(issue)

    current_count = sum(len(v) for v in catalog.values())

    while current_count < target_issue_types:
        category = random.choice(categories)
        base_issue = random.choice(BASE_CATALOG[category])
        clone_index = len(catalog[category]) + 1
        new_issue = clone_issue(base_issue, category, clone_index)
        catalog[category].append(new_issue)
        current_count += 1

    return catalog


ISSUE_CATALOG_LIST = generate_large_catalog(target_issue_types=5000)

print("Total categories:", len(ISSUE_CATALOG_LIST))
print("Total issue types:", sum(len(v) for v in ISSUE_CATALOG_LIST.values()))

Total categories: 8
Total issue types: 5000


In [6]:
ISSUE_CATALOG_LIST = generate_large_catalog(target_issue_types=5000)

print("Total categories:", len(ISSUE_CATALOG_LIST))
print("Total issue types:", sum(len(v) for v in ISSUE_CATALOG_LIST.values()))

Total categories: 8
Total issue types: 5000


In [7]:
def generate_synthetic_tickets(issue_catalog, num_tickets=2_000_000):
    tickets = []

    all_issues = []
    for category, issues in issue_catalog.items():
        for issue in issues:
            all_issues.append((category, issue))

    for i in range(1, num_tickets + 1):
        category, issue = random.choice(all_issues)

        subject = random.choice(issue["subject_templates"])
        description = random.choice(issue["description_templates"])

        ticket = {
            "ticket_id": f"TCKT_{i:07d}",
            "subject": subject,
            "description": description,
            "category": category,
            "issue_type": issue["issue_type"],
            "priority": issue["priority"],
            "resolution": issue["resolution"]
        }
        tickets.append(ticket)

    return pd.DataFrame(tickets)

In [8]:
df = generate_synthetic_tickets(ISSUE_CATALOG_LIST, num_tickets=2_000_000)
print(df.shape)
df.head()

(2000000, 7)


,ticket_id,subject,description,category,issue_type,priority,resolution
0,TCKT_0000001,Cannot authenticate to DB on postgres,The app cannot login to the database using con...,Database,Authentication to DB Failed Variant 398,High,Verify DB credentials and access permissions.
1,TCKT_0000002,Cannot resolve host on gateway,I am getting host resolution errors while brow...,Network,DNS Resolution Error Variant 270,Medium,Flush DNS cache and check network settings.
2,TCKT_0000003,Password prompt appears again and again in Out...,My mailbox is not syncing because Outlook repe...,Email,Outlook Password Prompt Loop Variant 107,High,Clear saved Outlook credentials and sign in ag...
3,TCKT_0000004,Laptop not connecting to WiFi on lan,My laptop cannot connect to office WiFi and ke...,Network,WiFi Not Connecting Variant 405,Medium,"Restart WiFi adapter, forget network, and reco..."
4,TCKT_0000005,Laptop keyboard not working on screen,I cannot type because keyboard input is failin...,Hardware,Keyboard Not Working Variant 508,Medium,Reconnect keyboard driver or restart the device.


In [9]:
def make_text_noisy(text):
    words = text.split()
    if len(words) <= 3:
        return text.lower()

    # random typo / short noisy version
    if random.random() < 0.5:
        random.shuffle(words)

    noisy = " ".join(words[:max(3, len(words)-random.randint(1, 3))])
    noisy = noisy.lower()

    replacements = {
        "password": "pass",
        "authentication": "auth",
        "connection": "conn",
        "problem": "prob",
        "working": "wrkng",
        "unable": "cant",
        "printer": "prntr",
        "database": "db"
    }

    for k, v in replacements.items():
        noisy = noisy.replace(k, v)

    return noisy

In [10]:
def generate_log_text(category, issue_type):
    log_templates = {
        "Network": [
            "ERROR 401: VPN auth failed",
            "DNS_PROBE_FINISHED_BAD_CONFIG",
            "WiFi adapter disconnected unexpectedly"
        ],
        "Email": [
            "Outlook auth popup loop detected",
            "Mailbox sync timeout error",
            "SMTP send failure code 550"
        ],
        "Hardware": [
            "BOOT_FAILURE: device did not start",
            "Keyboard driver not responding",
            "STOP_CODE: SYSTEM_SERVICE_EXCEPTION"
        ],
        "Software": [
            "Application crashed with exit code 1",
            "Installer failed due to missing dependency",
            "License server unreachable"
        ],
        "Access": [
            "Access denied for shared folder",
            "Account locked after failed attempts",
            "SSO token validation failed"
        ],
        "Printer": [
            "Printer offline status returned",
            "Print spooler service stuck",
            "Driver package missing"
        ],
        "Database": [
            "DB connection timeout after 30 seconds",
            "Database authentication failed",
            "Database service unavailable"
        ],
        "Security": [
            "MFA token delivery failed",
            "Antivirus update server not reachable",
            "Threat signature matched suspicious file"
        ]
    }

    if category in log_templates:
        return random.choice(log_templates[category])
    return f"System error related to {issue_type}"

In [11]:
def add_ticket_variations(df, noisy_ratio=0.30, screenshot_ratio=0.10, log_ratio=0.50):
    df = df.copy()

    df["is_noisy"] = False
    df["has_screenshot"] = False
    df["has_logs"] = False
    df["log_text"] = ""

    n = len(df)

    noisy_idx = np.random.choice(df.index, size=int(n * noisy_ratio), replace=False)
    screenshot_idx = np.random.choice(df.index, size=int(n * screenshot_ratio), replace=False)
    log_idx = np.random.choice(df.index, size=int(n * log_ratio), replace=False)

    # add noise
    for idx in noisy_idx:
        df.at[idx, "subject"] = make_text_noisy(str(df.at[idx, "subject"]))
        df.at[idx, "description"] = make_text_noisy(str(df.at[idx, "description"]))
        df.at[idx, "is_noisy"] = True

    # screenshot flag
    df.loc[screenshot_idx, "has_screenshot"] = True

    # logs
    for idx in log_idx:
        category = df.at[idx, "category"]
        issue_type = df.at[idx, "issue_type"]
        df.at[idx, "log_text"] = generate_log_text(category, issue_type)
        df.at[idx, "has_logs"] = True

    return df

In [12]:
df = add_ticket_variations(df)

print(df.shape)
print(df[["is_noisy", "has_screenshot", "has_logs"]].mean())
df.head()

(2000000, 11)
is_noisy          0.3
has_screenshot    0.1
has_logs          0.5
dtype: float64


,ticket_id,subject,description,category,issue_type,priority,resolution,is_noisy,has_screenshot,has_logs,log_text
0,TCKT_0000001,Cannot authenticate to DB on postgres,The app cannot login to the database using con...,Database,Authentication to DB Failed Variant 398,High,Verify DB credentials and access permissions.,False,False,False,
1,TCKT_0000002,Cannot resolve host on gateway,I am getting host resolution errors while brow...,Network,DNS Resolution Error Variant 270,Medium,Flush DNS cache and check network settings.,False,False,False,
2,TCKT_0000003,pass prompt appears again and again in outlook on,my syncing usage. not affecting outlook becaus...,Email,Outlook Password Prompt Loop Variant 107,High,Clear saved Outlook credentials and sign in ag...,True,False,True,Outlook auth popup loop detected
3,TCKT_0000004,Laptop not connecting to WiFi on lan,My laptop cannot connect to office WiFi and ke...,Network,WiFi Not Connecting Variant 405,Medium,"Restart WiFi adapter, forget network, and reco...",False,False,False,
4,TCKT_0000005,on wrkng laptop,is failing. affecting is type this i screen be...,Hardware,Keyboard Not Working Variant 508,Medium,Reconnect keyboard driver or restart the device.,True,False,False,


In [13]:
from PIL import Image, ImageDraw, ImageFont
from pathlib import Path
import os

In [14]:
def build_screenshot_text(row):
    parts = [str(row["subject"]), str(row["description"])]
    if row["has_logs"] and str(row["log_text"]).strip():
        parts.append("ERROR LOG: " + str(row["log_text"]))
    return "\n".join(parts)

In [15]:
def generate_synthetic_screenshots(df, output_dir="synthetic_screenshots", max_images=20000):
    output_path = Path(output_dir)
    output_path.mkdir(parents=True, exist_ok=True)

    df = df.copy()
    df["screenshot_path"] = ""

    screenshot_df = df[df["has_screenshot"] == True].copy()

    # runtime control: saari screenshot rows ki jagah limited images banao
    if len(screenshot_df) > max_images:
        screenshot_df = screenshot_df.sample(max_images, random_state=42)

    try:
        font = ImageFont.truetype("arial.ttf", 18)
    except:
        font = ImageFont.load_default()

    for i, (idx, row) in enumerate(screenshot_df.iterrows(), start=1):
        img = Image.new("RGB", (1000, 500), color="white")
        draw = ImageDraw.Draw(img)

        text = build_screenshot_text(row)
        text = text[:1200]   # image me bahut lamba text nahi
        draw.text((20, 20), text, fill="black", font=font)

        file_name = f"screenshot_{i}.png"
        file_path = output_path / file_name
        img.save(file_path)

        df.at[idx, "screenshot_path"] = str(file_path)

    return df

In [16]:
df = generate_synthetic_screenshots(df, output_dir="synthetic_screenshots", max_images=20000)

print(df.shape)
print("Screenshots generated:", (df["screenshot_path"] != "").sum())
df[["ticket_id", "has_screenshot", "screenshot_path"]].head()

KeyboardInterrupt: 

In [17]:
print("Screenshots generated so far:", (df["screenshot_path"] != "").sum() if "screenshot_path" in df.columns else 0)

Screenshots generated so far: 0


In [18]:
from pathlib import Path

screenshot_dir = Path("synthetic_screenshots")
files = sorted(screenshot_dir.glob("*.png"))

if "screenshot_path" not in df.columns:
    df["screenshot_path"] = ""

screenshot_idx = df[df["has_screenshot"] == True].index[:len(files)]

for idx, file_path in zip(screenshot_idx, files):
    df.at[idx, "screenshot_path"] = str(file_path)

print("Mapped screenshots:", (df["screenshot_path"] != "").sum())

Mapped screenshots: 19965


In [19]:
def build_model_input(row):
    parts = [
        str(row["subject"]).strip(),
        str(row["description"]).strip()
    ]

    if row["has_logs"] and str(row["log_text"]).strip():
        parts.append(str(row["log_text"]).strip())

    return " ".join(parts)

In [20]:
df["input_text"] = df.apply(build_model_input, axis=1)

print(df.shape)
print(df[["ticket_id", "input_text", "category", "issue_type"]].head())

(2000000, 13)
      ticket_id                                         input_text  category  \
0  TCKT_0000001  Cannot authenticate to DB on postgres The app ...  Database   
1  TCKT_0000002  Cannot resolve host on gateway I am getting ho...   Network   
2  TCKT_0000003  pass prompt appears again and again in outlook...     Email   
3  TCKT_0000004  Laptop not connecting to WiFi on lan My laptop...   Network   
4  TCKT_0000005  on wrkng laptop is failing. affecting is type ...  Hardware   

                                 issue_type  
0   Authentication to DB Failed Variant 398  
1          DNS Resolution Error Variant 270  
2  Outlook Password Prompt Loop Variant 107  
3           WiFi Not Connecting Variant 405  
4          Keyboard Not Working Variant 508  


In [21]:
os.makedirs("processed_data", exist_ok=True)
df.to_csv("processed_data/tickets_2M_final.csv", index=False)

print("Dataset saved successfully: processed_data/tickets_2M_final.csv")

Dataset saved successfully: processed_data/tickets_2M_final.csv
